# Mission d'adaptation : un nouveau site, sans code fourni
**ÉNONCÉ**

Vous savez maintenant scraper et crawler un site (Books to Scrape), et le faire poliment. **Cette mission change de site, et on ne vous donne aucun code de collecte.** C'est à vous d'adapter votre démarche à une structure différente.

**Le site :** `quotes.toscrape.com`, des citations célèbres, créé lui aussi pour s'entraîner.

## Ce que vous devez produire
1. **`citations`** : un tableau de **toutes** les citations (toutes les pages), avec les colonnes `texte`, `auteur`, `tags`, `url_page`, `url_auteur`.
2. **`auteurs`** : un tableau de la fiche de **chaque auteur, une seule fois**, avec `url_auteur`, `auteur`, `date_naissance`, `lieu_naissance`, `biographie`.
3. **`nb_requetes`** : le nombre de pages que votre script a demandées (pages de citations + fiches d'auteurs, sans compter `robots.txt`).
4. **`citations_completes`** : les citations enrichies de la fiche de leur auteur.
5. **`top_auteurs`** et **`top_tags`** : les 5 auteurs les plus cités et les 10 tags les plus fréquents.
6. **Une note de 5 lignes** : ce qui change par rapport à Books to Scrape, et comment vous l'avez géré.

## Règles
Les règles du notebook précédent s'appliquent : vous vous **présentez** (User-Agent avec contact), vous **lisez le robots.txt**, vous faites une **pause**. Ces outils vous sont fournis plus bas.

## Pour vous mettre sur la piste (sans code)
- **Inspectez** le site dans votre navigateur avant d'écrire quoi que ce soit. Qu'est-ce qui est pareil ? Qu'est-ce qui est différent ?
> Chaque citation est encapsulé dans un div class="quotes"
- Regardez **la forme des adresses** des liens « Next » et « (about) ».
> Le about est un lien relatif vers l'auteur
> Le next est dans un nav pager
- Un **même auteur** apparaît dans de nombreuses citations. Faut-il télécharger sa fiche à chaque fois ?
> Non
- Une citation a **plusieurs tags**. Comment stocker cela dans une seule case ?
> On concatene en un seul string, une liste, un dictionnaire, etc.
- Le texte d'une citation est entouré de **guillemets typographiques**. Faut-il les garder ?
> A priori non, mais on peut les garder pour le beau geste

In [2]:
import re, time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

SITE = "https://quotes.toscrape.com"      # un NOUVEAU site-école, à la structure différente de Books to Scrape

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Se présenter
Comme dans le notebook précédent : remplacez `___` par votre contact (adresse e-mail ou adresse d'une page de projet).

In [3]:
CONTACT = "carlos@papay.ou"
ENTETES = {"User-Agent": f"TPFormation/1.0 ({CONTACT}) python-requests"}   # TODO : remplacez ___ par votre contact
print(ENTETES)

{'User-Agent': 'TPFormation/1.0 (carlos@papay.ou) python-requests'}


In [4]:
# Contrôle (ne pas modifier)
ua = ENTETES['User-Agent']
controle(('@' in ua or 'http' in ua) and '___' not in ua and not any(m in ua.lower() for m in ('example', 'exemple', 'votre.')), 'User-Agent renseigné : ' + ua, 'Remplacez ___ par un vrai contact')

✅ User-Agent renseigné : TPFormation/1.0 (carlos@papay.ou) python-requests


## Étape 2 — Vos outils (fournis)
La fonction de requête polie et la lecture du `robots.txt`, issues du notebook précédent.

In [5]:
PAUSE = 1.0

def get_poli(url, essais=3):
    for n in range(essais):
        time.sleep(PAUSE)
        reponse = requests.get(url, headers=ENTETES, timeout=20)
        if reponse.status_code in (429, 503):
            time.sleep(2 ** (n + 1))
            continue
        return reponse
    raise RuntimeError("Échec après " + str(essais) + " essais : " + url)

def lire_robots(base):
    reponse = requests.get(base + "/robots.txt", headers=ENTETES)
    print(base, "-> code", reponse.status_code)
    if reponse.status_code == 404:
        print("   pas de fichier robots.txt : aucune restriction annoncée")
        return None
    if reponse.status_code != 200:
        raise RuntimeError("Réponse inattendue pour robots.txt : on s'abstient par prudence")
    regles = RobotFileParser(); regles.parse(reponse.text.splitlines())
    print(reponse.text[:500])
    return regles

## Étape 3 — Consulter le robots.txt
Lisez le `robots.txt` de ce site, puis demandez s'il est permis de visiter la page 2 des citations (`/page/2/`) et la fiche d'un auteur (`/author/Albert-Einstein`). Stockez les deux réponses (vrai ou faux) dans `autorise_pages` et `autorise_auteur`.

In [6]:
def getAutorisation(baseSite, listeChemins):
    autorise = {}
    regles = lire_robots(baseSite)
    for chemin in listeChemins:
        reponse_regle = True if regles is None else regles.can_fetch(ua, urljoin(baseSite, chemin))
        autorise[chemin] = reponse_regle
        print("  ", "OUI" if reponse_regle else "NON", chemin)
    return autorise

In [7]:
autorisations = getAutorisation(SITE, ["/page/2/", "/author/Albert-Einstein"])
autorise_pages = autorisations["/page/2/"]
autorise_auteur = autorisations["/author/Albert-Einstein"]

https://quotes.toscrape.com -> code 404
   pas de fichier robots.txt : aucune restriction annoncée
   OUI /page/2/
   OUI /author/Albert-Einstein


In [8]:
# Contrôle (ne pas modifier)
controle(isinstance(autorise_pages, bool) and isinstance(autorise_auteur, bool), 'Autorisations établies : pages=%s, auteur=%s' % (autorise_pages, autorise_auteur), 'Attendu : deux booléens')

✅ Autorisations établies : pages=True, auteur=True


## Étape 4 — Collecter toutes les citations *(à écrire seul)*
Objectif : la variable **`citations`** (un `DataFrame`) et le compteur **`nb_requetes`**. Cette cellule est **vide** : à vous d'écrire la boucle.

In [9]:
# Colonnes attendues : texte, auteur, tags, url_page, url_auteur
def lire_citation(quote, url_page):
    lien_auteur = quote.select_one("small.author ~ a")
    return {
        "texte": quote.select_one("span.text").get_text(strip=True).strip("“”"),
        "auteur": quote.select_one("small.author").get_text(strip=True),
        "tags": [tag.get_text(strip=True) for tag in quote.select("a.tag")],
        "url_page": url_page,
        "url_auteur": urljoin(url_page, lien_auteur["href"]),
    }

In [10]:
# À vous : collecter toutes les citations (variables attendues : citations, nb_requetes)
debut = time.time()
url, nb_citations, nb_requetes, list_citations = SITE + "/", 0, 0, []
while url:
    reponse = get_poli(url)
    nb_requetes += 1
    page = BeautifulSoup(reponse.content, "html.parser")
    for quote in page.select('div.quote'): # pour chaque citation au sein de la page
        list_citations.append(lire_citation(quote,url))
        nb_citations += 1
    suivant = page.select_one("li.next a")
    url = urljoin(url, suivant["href"]) if suivant else None
print(f"Version polie : {nb_citations} citations, {nb_requetes} requêtes, {time.time() - debut:.1f} secondes")

citations = pd.DataFrame(list_citations)

Version polie : 100 citations, 10 requêtes, 15.0 secondes


In [11]:
# Contrôle (ne pas modifier)
colonnes = {"texte", "auteur", "tags", "url_page", "url_auteur"}
controle(isinstance(citations, pd.DataFrame) and colonnes <= set(citations.columns) and len(citations) >= 10,
         "%d citations collectées, colonnes correctes" % len(citations), "Attendu : un DataFrame avec les colonnes " + ", ".join(sorted(colonnes)) + " et au moins 10 lignes")
controle(citations["tags"].map(lambda t: isinstance(t, list)).all() and not citations["texte"].str.startswith(("“", '"')).any(),
         "Les tags sont des listes et les guillemets typographiques sont retirés", "Les tags doivent être des listes, et le texte ne doit plus commencer par un guillemet")
controle(citations["url_auteur"].str.startswith("http").all() and citations["url_page"].str.startswith("http").all(),
         "Toutes les adresses sont complètes", "Les adresses doivent commencer par http : pensez aux liens relatifs")
controle(not citations.duplicated(subset=["texte", "auteur"]).any(), "Aucune citation en double", "Il y a des citations en double : une page a-t-elle été lue deux fois ?")

✅ 100 citations collectées, colonnes correctes
✅ Les tags sont des listes et les guillemets typographiques sont retirés
✅ Toutes les adresses sont complètes
✅ Aucune citation en double


## Étape 5 — Collecter la fiche de chaque auteur *(à écrire seul)*
Objectif : la variable **`auteurs`**, avec **une seule ligne par auteur**. Inspectez une fiche d'auteur pour trouver où sont la date et le lieu de naissance, et la biographie. **Attention au nombre de requêtes** : le contrôle suivant le vérifie.

In [12]:
# Colonnes attendues : `url_auteur`, `auteur`, `date_naissance`, `lieu_naissance`, `biographie`
def lire_auteur(auteur, url_page):
    return {
        "url_auteur": url_page,
        "auteur": auteur.select_one("h3.author-title").get_text(strip=True),
        "date_naissance": auteur.select_one("span.author-born-date").get_text(strip=True),
        "lieu_naissance": auteur.select_one("span.author-born-location").get_text(strip=True),
        "biographie": auteur.select_one("div.author-description").get_text(strip=True),
    }

In [13]:
# À vous : collecter une fiche par auteur (variable attendue : auteurs)

debut = time.time()
list_auteurs, nb_requetes, nb_auteurs = [], 0, 0

for url_auteur in citations["url_auteur"].unique():
    reponse = get_poli(url_auteur)
    nb_requetes += 1
    page = BeautifulSoup(reponse.content, "html.parser")
    auteur = page.select_one("div.author-details")
    list_auteurs.append(lire_auteur(auteur, url_auteur))
    nb_auteurs += 1

print(f"Version polie : {nb_auteurs} auteurs, {nb_requetes} requêtes, {time.time() - debut:.1f} secondes")

auteurs = pd.DataFrame(list_auteurs)

display(auteurs.sample(5))

Version polie : 50 auteurs, 50 requêtes, 80.3 secondes


,url_auteur,auteur,date_naissance,lieu_naissance,biographie
3,https://quotes.toscrape.com/author/Marilyn-Monroe,Marilyn Monroe,"June 01, 1926",in The United States,Marilyn Monroe (born Norma Jeane Mortenson; Ju...
17,https://quotes.toscrape.com/author/Mother-Teresa,Mother Teresa,"August 26, 1910","in Skopje, Macedonia, the Former Yugoslav Repu...","Blessed Teresa of Calcutta, or Mother Teresa, ..."
19,https://quotes.toscrape.com/author/Jim-Henson,Jim Henson,"September 24, 1936","in Greenville, Mississippi, The United States","James Maury ""Jim"" Henson was the most widely k..."
47,https://quotes.toscrape.com/author/Khaled-Hoss...,Khaled Hosseini,"March 04, 1965","in Kabul, Afghanistan","Hosseini was born in Kabul, Afghanistan, in 19..."
31,https://quotes.toscrape.com/author/Ernest-Hemi...,Ernest Hemingway,"July 21, 1899","in Oak Park, Illinois, The United States",Ernest Miller Hemingway was an American author...


In [14]:
# Contrôle (ne pas modifier)
colonnes_a = {"url_auteur", "auteur", "date_naissance", "lieu_naissance", "biographie"}
controle(isinstance(auteurs, pd.DataFrame) and colonnes_a <= set(auteurs.columns) and auteurs["url_auteur"].is_unique,
         "%d fiches d'auteurs, une seule par auteur" % len(auteurs), "Attendu : les colonnes " + ", ".join(sorted(colonnes_a)) + " et une seule ligne par auteur")
controle(set(citations["url_auteur"]) <= set(auteurs["url_auteur"]), "Tous les auteurs cités ont une fiche", "Certains auteurs cités n'ont pas de fiche")
attendu = citations["url_page"].nunique() + citations["url_auteur"].nunique()
controle(nb_requetes <= attendu + 2, "%d requêtes pour %d attendues au minimum : fiches non redemandées" % (nb_requetes, attendu), "%d requêtes alors que %d suffisent : un auteur a été redemandé plusieurs fois" % (nb_requetes, attendu))

✅ 50 fiches d'auteurs, une seule par auteur
✅ Tous les auteurs cités ont une fiche
✅ 50 requêtes pour 60 attendues au minimum : fiches non redemandées


## Étape 6 — Fusionner et analyser *(à écrire seul)*
Objectifs : **`citations_completes`** (chaque citation avec la fiche de son auteur), **`top_auteurs`** (les 5 auteurs les plus cités) et **`top_tags`** (les 10 tags les plus fréquents). Indice : pour compter des tags stockés dans des listes, il faut d'abord « déplier » ces listes.

In [44]:
# À vous : citations_completes, top_auteurs, top_tags

citations_completes = pd.merge(citations, auteurs[['url_auteur', 'date_naissance', 'lieu_naissance', 'biographie']], on="url_auteur", how="left")

display(citations_completes.sample(5))

top_auteurs = citations['auteur'].value_counts().head(5)

display(top_auteurs)

top_tags = citations.explode('tags')['tags'].value_counts().head(10)

display(top_tags)



,texte,auteur,tags,url_page,url_auteur,date_naissance,lieu_naissance,biographie
11,It takes a great deal of bravery to stand up t...,J.K. Rowling,"[courage, friends]",https://quotes.toscrape.com/page/2/,https://quotes.toscrape.com/author/J-K-Rowling,"July 31, 1965","in Yate, South Gloucestershire, England, The U...",See also: Robert GalbraithAlthough she writes ...
9,"A day without sunshine is like, you know, night.",Steve Martin,"[humor, obvious, simile]",https://quotes.toscrape.com/,https://quotes.toscrape.com/author/Steve-Martin,"August 14, 1945","in Waco, Texas, The United States","Stephen Glenn ""Steve"" Martin is an American ac..."
40,A reader lives a thousand lives before he dies...,George R.R. Martin,"[read, readers, reading, reading-books]",https://quotes.toscrape.com/page/5/,https://quotes.toscrape.com/author/George-R-R-...,"September 20, 1948","in Bayonne, New Jersey, The United States","George R. R. Martin was born September 20, 194..."
32,"The truth is, everyone is going to hurt you. Y...",Bob Marley,[friendship],https://quotes.toscrape.com/page/4/,https://quotes.toscrape.com/author/Bob-Marley,"February 06, 1945","in Nine Mile, Saint Ann, Jamaica","Robert ""Bob"" Nesta Marley OM was a Jamaican si..."
45,The real lover is the man who can thrill you b...,Marilyn Monroe,[love],https://quotes.toscrape.com/page/5/,https://quotes.toscrape.com/author/Marilyn-Monroe,"June 01, 1926",in The United States,Marilyn Monroe (born Norma Jeane Mortenson; Ju...


auteur
Albert Einstein    10
J.K. Rowling        9
Marilyn Monroe      7
Dr. Seuss           6
Mark Twain          6
Name: count, dtype: int64

tags
love             14
inspirational    13
life             13
humor            12
books            11
reading           7
friendship        5
friends           4
truth             4
simile            3
Name: count, dtype: int64

In [43]:
# Contrôle (ne pas modifier)
controle(len(citations_completes) == len(citations) and "date_naissance" in citations_completes.columns and citations_completes["date_naissance"].notna().all(),
         "Fusion correcte : chaque citation a la fiche de son auteur", "La fusion doit garder toutes les citations, chacune avec la date de naissance de son auteur")
controle(len(top_auteurs) == min(5, citations["auteur"].nunique()) and len(top_tags) == min(10, citations.explode("tags")["tags"].nunique()) and top_auteurs.is_monotonic_decreasing and top_tags.is_monotonic_decreasing,
         "Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent", "Attendu : 5 auteurs et 10 tags triés par fréquence décroissante")

✅ Fusion correcte : chaque citation a la fiche de son auteur
✅ Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent


## Étape 7 — Votre note (5 lignes)
Répondez ici, en 5 lignes maximum :

1. Ce qui était **identique** à Books to Scrape.
> La logique intraseque au scrapping via beautifulSoup, recup des données via les balises html
2. Ce qui était **différent**.
> La structure html du site
3. Ce que vous avez dû **changer** dans votre démarche.
> J'ai dû changer la manière de récupérer les données des auteurs, nouveaux format de balises
4. Comment vous avez évité de **redemander** les fiches d'auteurs.
> En bouclant sur les url_auteurs unique
5. Ce qui, à votre avis, **casserait** votre script si le site changeait.
> Un changement de nomenclautre dans les balises html

> 